# 09. 기하학적 변환

> 강의 실습 노트북 `translation.ipynb`와 `namecard.ipynb`를 합쳤습니다. 명함 파일 경로만 배포용 상대 경로로 바꿨습니다.

강의 화면의 코드 순서와 파일명을 유지했습니다. 데이터 파일은 코드에 표시된 `./data` 또는 `../data` 상대 경로에 두세요.


## 이동 변환


In [ ]:
import sys
import numpy as np
import cv2

src = cv2.imread('./data/tekapo.bmp')

aff = np.array([[1, 0, 200],
                [0, 1, 100]], dtype=np.float32)    # x로 200, y로 100 이동

dst = cv2.warpAffine(src, aff, (0, 0))             # 출력 크기 = 입력과 동일

cv2.imshow('src', src)
cv2.imshow('dst', dst)
cv2.waitKey()
cv2.destroyAllWindows()


## 전단 변환


In [ ]:
src = cv2.imread("./data/tekapo.bmp")

aff = np.array([[1, 0.5, 0],
                [0, 1,   0]], dtype=np.float32)     # 가로 방향 전단, m = 0.5

h, w = src.shape[:2]

dst = cv2.warpAffine(src, aff, (w + int(h * 0.5), h))   # 밀린 만큼 폭을 늘림

cv2.imshow("src", src)
cv2.imshow("dst", dst)
cv2.waitKey()
cv2.destroyAllWindows()


## 확대와 축소


In [ ]:
import sys
import numpy as np
import cv2

src = cv2.imread('./data/rose.bmp')       # src.shape = (320, 480)

if src is None:
    print('Image load failed!')
    sys.exit()

dst1 = cv2.resize(src, (0, 0), fx=4, fy=4, interpolation=cv2.INTER_NEAREST)
dst2 = cv2.resize(src, (1920, 1280))                                    # INTER_LINEAR
dst3 = cv2.resize(src, (1920, 1280), interpolation=cv2.INTER_CUBIC)
dst4 = cv2.resize(src, (1920, 1280), interpolation=cv2.INTER_LANCZOS4)

cv2.imshow('src', src)
cv2.imshow('dst1', dst1[500:900, 400:800])     # 같은 영역만 잘라 비교
cv2.imshow('dst2', dst2[500:900, 400:800])
cv2.imshow('dst3', dst3[500:900, 400:800])
cv2.imshow('dst4', dst4[500:900, 400:800])
cv2.waitKey()
cv2.destroyAllWindows()


## 회전행렬 직접 계산


In [ ]:
import math

src = cv2.imread("./data/tekapo.bmp")

rad = 20 * math.pi / 180                        # degree → radian
aff = np.array([[ math.cos(rad), math.sin(rad), 0],
                [-math.sin(rad), math.cos(rad), 0]], dtype=np.float32)

dst = cv2.warpAffine(src, aff, (0, 0))

cv2.imshow("src", src)
cv2.imshow("dst", dst)
cv2.waitKey()
cv2.destroyAllWindows()


## 중심 기준 회전


In [ ]:
import sys
import numpy as np
import cv2

src = cv2.imread("./data/tekapo.bmp")

if src is None:
    print("Image load failed!")
    sys.exit()

cp = (src.shape[1] / 2, src.shape[0] / 2)        # 영상 중앙 (x, y)
rot = cv2.getRotationMatrix2D(cp, 20, 0.7)       # 20도 회전 + 0.7배 축소

dst = cv2.warpAffine(src, rot, (0, 0))

cv2.imshow("src", src)
cv2.imshow("dst", dst)
cv2.waitKey()

cv2.destroyAllWindows()


## 회전 결과 확인


In [ ]:
rot


## 명함 실습 라이브러리


In [ ]:
import sys
import cv2
import numpy as np


## 명함 이미지 읽기


In [ ]:
src = cv2.imread("./data/pinkwink_namecard.png")


## 원본 크기


In [ ]:
src.shape


## 미리보기 축소


In [ ]:
dst = cv2.resize(src, (0, 0), fx=0.5, fy=0.5)     # 0.5배 축소


## 축소 크기


In [ ]:
dst.shape


## 선택점과 미리보기


In [ ]:
points = []
preview = cv2.resize(src, (0, 0), fx=0.5, fy=0.5)


## 마우스 콜백


In [ ]:
def click_event(event, x, y, flags, param):
    if event == cv2.EVENT_LBUTTONDOWN and len(points) < 4:
        points.append([x, y])

        cv2.imshow("preview", preview)
        print(f"{len(points)}번 좌표: ({x}, {y})")


## 네 점 선택과 투시 변환


In [ ]:
cv2.imshow("preview", preview)
cv2.setMouseCallback("preview", click_event)

while True:
    if cv2.waitKey() == 27:          # ESC로 종료
        break

cv2.destroyAllWindows()

if len(points) != 4:
    raise ValueError("좌표를 4개 선택해야 합니다.")

# 클릭 순서: 왼쪽 위 → 오른쪽 위 → 오른쪽 아래 → 왼쪽 아래
src_quad = np.float32(points)

w, h = 720, 480
dst_quad = np.float32([[0, 0],[w - 1, 0],[w - 1, h - 1],[0, h - 1]])

pers = cv2.getPerspectiveTransform(src_quad, dst_quad)

result = cv2.warpPerspective(preview, pers, (w, h))

cv2.imshow("result", result)
cv2.waitKey()
cv2.destroyAllWindows()
